#### Sparse/ Dense Combination Statistics
- Data:
    - Sparse: splade_dev.trec
    - Dense: splade_llama_rerank.tsv
- Alpha corresponds to sparse weight, 1 - alpha corresponds to dense weight

In [38]:
import pandas as pd
%run best_weights.ipynb

#### Weight Function

In [39]:
def best_weights(dense_df, sparse_df, query_id, alpha_values):
    best_alpha = 0
    best_mrr = 0

    for alpha in alpha_values:
        # Filter by query ID
        filtered_dense_df = dense_df[dense_df["Query ID"] == query_id].copy()
        filtered_sparse_df = sparse_df[sparse_df["Query ID"] == query_id].copy()

        # Find weighted scores
        filtered_dense_df["Score"] *= (1 - alpha)
        filtered_sparse_df["Score"] *= alpha

        # Merge rankings
        merged = filtered_dense_df.merge(filtered_sparse_df, on="Document ID", how="outer", suffixes=("_dense", "_sparse")).fillna(0)
        merged["Final Score"] = merged["Score_dense"] + merged["Score_sparse"]

        # Rank documents
        ranked_results = merged.sort_values("Final Score", ascending=False)
        ranked_docs = ranked_results["Document ID"].tolist()

        # MRR
        relevant_doc = qrels_df[qrels_df["Query ID"] == query_id]["Document ID"].iloc[0]
        if relevant_doc not in ranked_docs:
            mrr_score = 0
        else:
            rank = ranked_docs.index(relevant_doc) + 1
            if rank > 10:
                mrr_score = 0
            else:
                mrr_score = 1 / rank

        # Update alpha and MRR
        if mrr_score > best_mrr:
            best_mrr = mrr_score
            best_alpha = alpha

    return best_alpha, best_mrr


#### Weight of 0.5

In [ ]:
mrr_scores = []

for query in query_ids:
    mrr_score = best_weights(dense_df, splade_df, query, [0.5])
    mrr_scores.append(mrr_score[1])

# print(mrr_scores)

[1.0, 0.25, 1.0, 0.25, 0, 1.0, 0, 1.0, 0.5, 1.0, 0.5, 0.16666666666666666, 0, 0, 0, 0.5, 0, 0, 1.0, 0, 0.16666666666666666, 1.0, 0.14285714285714285, 0.2, 0.125, 0.16666666666666666, 0.3333333333333333, 0.1111111111111111, 0.125, 1.0, 0.2, 0, 0, 0, 0.16666666666666666, 0, 0, 1.0, 0.125, 1.0, 0.2, 1.0, 0, 1.0, 0.2, 1.0, 0.3333333333333333, 0, 0, 0.5, 0, 0.16666666666666666, 0, 1.0, 1.0, 1.0, 0, 0, 0, 0, 0.1111111111111111, 1.0, 1.0, 0.3333333333333333, 1.0, 0, 0.3333333333333333, 0, 0.2, 1.0, 1.0, 0.16666666666666666, 0.25, 0, 0.5, 0.5, 0.16666666666666666, 1.0, 1.0, 0, 1.0, 0, 1.0, 1.0, 1.0, 1.0, 0.25, 0, 0, 1.0, 0, 0, 0.5, 0, 0, 0.5, 0, 1.0, 1.0, 0, 0, 0.2, 0.3333333333333333, 0, 0, 0, 0.125, 0.16666666666666666, 0, 1.0, 0.25, 0.2, 0.3333333333333333, 1.0, 0.3333333333333333, 1.0, 0.16666666666666666, 0.25, 0, 0, 0.1, 1.0, 1.0, 0, 1.0, 0.3333333333333333, 1.0, 1.0, 1.0, 0, 1.0, 0.5, 0.2, 0, 0.125, 0.16666666666666666, 0.25, 0.3333333333333333, 0.5, 0.125, 1.0, 1.0, 0.125, 0, 0.5, 0.16

In [41]:
print("Weight of 0.5")
print("----------------")
print("Min MRR:", min(mrr_scores))
print("Max MRR:", max(mrr_scores))
print("Average MRR:", sum(mrr_scores)/len(mrr_scores))

Weight of 0.5
----------------
Min MRR: 0
Max MRR: 1.0
Average MRR: 0.3853263291945242


#### Optimal Weights

In [42]:
best_alphas = []
best_mrrs = []

for query in query_ids:
    best_alpha, best_mrr = best_weights(dense_df, splade_df, query, alpha_values)
    best_alphas.append(best_alpha)
    best_mrrs.append(best_mrr)

In [46]:
print("Optimal Weight")
print("----------------")
print("Min MRR:", min(best_mrrs))
print("Max MRR:", max(best_mrrs))
print("Average MRR:", sum(best_mrrs)/len(best_mrrs))
print("----------------")
print("Min Alpha:", min(best_alphas))
print("Max Alpha:", max(best_alphas))

avg_alpha = sum(best_alphas)/len(best_alphas)
print("Average Alpha:", avg_alpha)


Optimal Weight
----------------
Min MRR: 0
Max MRR: 1.0
Average MRR: 0.47861600036385243
----------------
Min Alpha: 0.0
Max Alpha: 1.0
Average Alpha: 0.00250716332378223


In [52]:
import csv

best_weights = {'Query ID': query_ids, 'Best Alpha': best_alphas, 'MRR': best_mrrs}

with open("best_weights_mrr10.csv", "w") as outfile:
	writer = csv.writer(outfile)
	
	# convert dict keys to a list
	key_list = list(best_weights.keys())
	
	writer.writerow(best_weights.keys())
	
	# iterate each column and assign corresponding values to each column
	for i in range(6980):
		writer.writerow([best_weights[x][i] for x in key_list])


#### Average of Optimal Weights

In [49]:
mrr_scores = []

for query in query_ids:
    mrr_score = best_weights(dense_df, splade_df, query, [avg_alpha])
    mrr_scores.append(mrr_score[1])

# print(mrr_scores)

[1.0, 0.25, 1.0, 0.25, 0, 1.0, 0, 1.0, 0.5, 1.0, 0.5, 0.16666666666666666, 0, 0, 0, 0.5, 0, 0, 1.0, 0, 0.16666666666666666, 1.0, 0.14285714285714285, 0.2, 0.125, 0.16666666666666666, 0.3333333333333333, 0.1111111111111111, 0.125, 1.0, 0.2, 0, 0, 0, 0.16666666666666666, 0, 0, 1.0, 0.125, 1.0, 0.2, 1.0, 0, 1.0, 0.2, 1.0, 0.3333333333333333, 0, 0, 0.5, 0, 0.16666666666666666, 0, 1.0, 1.0, 1.0, 0, 0, 0, 0, 0.1111111111111111, 1.0, 1.0, 0.3333333333333333, 1.0, 0, 0.3333333333333333, 0, 0.2, 1.0, 1.0, 0.16666666666666666, 0.25, 0, 0.5, 0.5, 0.16666666666666666, 1.0, 1.0, 0, 1.0, 0, 1.0, 1.0, 1.0, 1.0, 0.25, 0, 0, 1.0, 0, 0, 0.5, 0, 0, 0.5, 0, 1.0, 1.0, 0, 0, 0.2, 0.3333333333333333, 0, 0, 0, 0.125, 0.16666666666666666, 0, 1.0, 0.25, 0.2, 0.3333333333333333, 1.0, 0.3333333333333333, 1.0, 0.16666666666666666, 0.25, 0, 0, 0.1, 1.0, 1.0, 0, 1.0, 0.3333333333333333, 1.0, 1.0, 1.0, 0, 1.0, 0.5, 0.2, 0, 0.125, 0.16666666666666666, 0.25, 0.3333333333333333, 0.5, 0.125, 1.0, 1.0, 0.125, 0, 0.5, 0.16

In [50]:
print("Average of Optimal Weights")
print("----------------")
print("Min MRR:", min(mrr_scores))
print("Max MRR:", max(mrr_scores))
print("Average MRR:", sum(mrr_scores)/len(mrr_scores))

Average of Optimal Weights
----------------
Min MRR: 0
Max MRR: 1.0
Average MRR: 0.385513485241279
